In [ ]:
from collections import defaultdict
from pathlib import Path

import polars as pl
from Bio import SeqIO
from ete4 import Tree

In [ ]:
CWD = Path().resolve()
BASE = CWD.parent.parent

print(f"Current working directory: {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

# The NCBI tree's branch lengths come from to_ultrametric() and carry no divergence
# information. PRANK reads branch lengths as substitutions/site, so the subgroup tree
# is rescaled to this root-to-tip depth. None keeps the lengths as they are.
TREE_DEPTH = 0.1

# Branch length for sister tips added when one taxon has several IGRs in a gene pair.
DUP_BRANCH = 1e-4

In [ ]:
def rescale(tree: Tree, depth: float) -> None:
    """Scale all branch lengths in place so the deepest tip sits at `depth` from the root."""
    node_depth = {}
    for node in tree.traverse("preorder"):
        parent_depth = node_depth[node.up] if node.up else 0.0
        branch = (node.dist or 0.0) if node.up else 0.0  # ignore any root length
        node_depth[node] = parent_depth + branch
    max_depth = max(node_depth[leaf] for leaf in tree.leaves())
    if max_depth > 0:
        factor = depth / max_depth
        for node in tree.traverse():
            if node.dist:
                node.dist *= factor


def subgroup_tree(group_tree: Tree, taxa: list[str]) -> Tree:
    """Group tree pruned to `taxa`, bifurcating, rescaled."""
    tree = group_tree.copy()
    tree.prune(taxa, preserve_branch_length=True)
    tree.resolve_polytomy()  # PRANK needs a bifurcating tree; new branches get length 0
    if TREE_DEPTH is not None:
        rescale(tree, TREE_DEPTH)
    return tree


def gene_pair_tree(subg_tree: Tree, ids_by_taxon: dict[str, list[str]]) -> Tree:
    """Subgroup tree pruned to the taxa present, tips relabelled to IGR IDs."""
    tree = subg_tree.copy()
    tree.prune(list(ids_by_taxon), preserve_branch_length=True)

    for leaf in list(tree.leaves()):
        ids = ids_by_taxon[leaf.name]
        if len(ids) == 1:
            leaf.name = ids[0]
            continue
        # several IGRs from one taxon: turn the tip into a node with one child per IGR
        leaf.name = ""
        for igr_id in ids:
            leaf.add_child(name=igr_id, dist=DUP_BRANCH)

    tree.resolve_polytomy()  # 3+ copies from one taxon would be a polytomy
    return tree

In [ ]:
for g, subgs in groups_map.items():
    group_tree_file = BASE / g / "tree.nwk"
    if not group_tree_file.exists():
        print(f"{g}: no tree.nwk, skipping group")
        continue
    group_tree = Tree(group_tree_file.read_text(), parser=1)
    group_taxa = {leaf.name for leaf in group_tree.leaves()}

    for subg in subgs:
        subg_dir = BASE / g / subg
        print(f"Processing {g}/{subg}")

        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        if not gp_file.exists():
            print(f"  no {gp_file.name}, skipping")
            continue

        gp_dir = subg_dir / "gene_pair_fastas"
        tree_dir = subg_dir / "gene_pair_trees"
        tree_dir.mkdir(parents=True, exist_ok=True)

        # --- subgroup tree (taxid tips)
        tsv = pl.read_csv(subg_dir / f"{subg}.tsv", separator="\t")
        subg_taxa = set(tsv["ncbi_taxid"].cast(pl.String))
        missing = sorted(subg_taxa - group_taxa)
        if missing:
            print(f"  {len(missing)} subgroup taxa not in the group tree: {missing}")

        subg_tree = subgroup_tree(group_tree, sorted(subg_taxa & group_taxa))
        subg_tree.write(outfile=str(subg_dir / "tree.nwk"), parser=5)
        print(f"  subgroup tree: {len(list(subg_tree.leaves()))} tips -> tree.nwk")

        # --- IGR ID -> taxid, via AN
        atlas = pl.read_csv(subg_dir / f"{subg}_summary_atlas.tsv", separator="\t")
        id_to_taxon = dict(
            atlas.select("ID", "AN")
            .join(tsv.select("AN", "ncbi_taxid"), on="AN", how="inner")
            .select(pl.col("ID").cast(pl.String), pl.col("ncbi_taxid").cast(pl.String))
            .iter_rows()
        )
        tree_taxa = {leaf.name for leaf in subg_tree.leaves()}

        # --- one tree per gene pair (IGR ID tips)
        gp_df = pl.read_csv(gp_file, separator="\t")
        n_written, n_dup_taxa, problems = 0, 0, []
        for gp in gp_df["gene_pair"]:
            fasta = gp_dir / f"gp_{gp}.fasta"
            if not fasta.exists():
                problems.append(f"{gp}: missing FASTA")
                continue

            ids_by_taxon = defaultdict(list)
            unplaced = []
            for rec in SeqIO.parse(fasta, "fasta"):
                taxon = id_to_taxon.get(rec.id)
                if taxon in tree_taxa:
                    ids_by_taxon[taxon].append(rec.id)
                else:
                    unplaced.append(rec.id)

            if unplaced:
                # PRANK fails if a FASTA sequence has no tip, so don't write a partial tree
                problems.append(
                    f"{gp}: {len(unplaced)} IGRs without a tip ({unplaced[:3]}...)"
                )
                continue
            if len(ids_by_taxon) < 2:
                problems.append(f"{gp}: fewer than 2 taxa")
                continue

            n_dup_taxa += sum(len(ids) > 1 for ids in ids_by_taxon.values())
            gene_pair_tree(subg_tree, ids_by_taxon).write(
                outfile=str(tree_dir / f"gp_{gp}.nwk"), parser=5
            )
            n_written += 1

        print(
            f"  gene-pair trees: {n_written}/{gp_df.height} written"
            f" | taxa with several IGRs (sister tips): {n_dup_taxa}"
        )
        for p in problems:
            print(f"    {p}")